# imports

In [26]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

In [27]:
df = pd.read_csv(r"D:\project\laptop_pricing_intelligence_pipeline\data\raw\raw_laptops.csv")
df.head(5)

,title,link,current_price,old_price,discount,rating,rating_num,shipping,Best Seller Ranking,Series,...,CPU,Screen,Storage,Consumer Alert,Docking Connector,Software Included,Graphics Card,Communication,Other Input Devices,Virtual Reality Ready
0,"Asus VivoBook 16"" Copilot+ PC Laptop AMD Ryzen...",https://www.newegg.com/asus-vivobook-16-0-amd-...,$849.99,NaN,NaN,4.4,130.0,Free Shipping,#1 in All Laptop,VivoBook,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Acer Aspire Go 15 Laptop - AMD Ryzen 7 5825U -...,https://www.newegg.com/acer-america-aspire-go-...,$599.99,$749.99,20%,4.4,97.0,Free Shipping,#2 in All Laptop,Aspire Go 15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"ASUS Vivobook S 16"" 3K OLED Intel Core Ultra 7...",https://www.newegg.com/asus-vivobook-16-2880x1...,"$1,299.99","$1,599.99",18%,4.5,11.0,Free Shipping,#3 in All Laptop,VivoBook S16,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MSI Venture 16 AI Touchscreen Laptop Intel Cor...,https://www.newegg.com/msi-venture-16-ai-16-fh...,"$1,029.00","$1,199.99",14%,4.5,10.0,Free Shipping,#4 in All Laptop,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Acer Aspire Go 15 15.6"" - AMD Ryzen 7 5825U - ...",https://www.newegg.com/acer-america-aspire-go-...,$674.99,$699.99,NaN,4.4,97.0,Free Shipping,#5 in All Laptop,Aspire Go 15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
df.loc[331, ['title', 'Brand']]

title    Lenovo ThinkPad T14 Business AI Laptop, 14" FH...
Brand                                                   HP
Name: 331, dtype: str

In [29]:
# Filter non-laptop items (e.g. USB-C Docking Stations)
df = df[~df['title'].str.contains('USB-C DOCKING STATION', case=False, na=False)].copy().reset_index(drop=True)
print(f"Filtered Laptop Dataset: {len(df)} rows")

Filtered Laptop Dataset: 719 rows


In [30]:
df.loc[331, ['title', 'Brand']]

title    Lenovo V15 15.6 FHD Business Laptop - AMD Ryze...
Brand                                                  NaN
Name: 331, dtype: str

In [31]:
df.columns

Index(['title', 'link', 'current_price', 'old_price', 'discount', 'rating',
       'rating_num', 'shipping', 'Best Seller Ranking', 'Series',
       'AI Features', 'Operating System', 'Processor Name', 'Screen Size',
       'Resolution', 'Refresh Rate', 'SSD', 'Memory', 'Memory Slot (Total)',
       'WiFi Generation', 'Bluetooth', 'USB', 'HDMI', 'Other port',
       'Backlit Keyboard', 'Battery', 'First Listed on Newegg', 'Brand',
       'Model', 'Part Number', 'Color', 'CPU Type', 'Number of Cores',
       'CPU L2 Cache', 'CPU L3 Cache', 'Touchscreen', 'Wide Screen Support',
       'Display Type', 'Panel', 'GPU/VPU', 'Video Memory', 'Graphic Type',
       'Storage Spec', 'Memory Speed', 'Memory Spec',
       'Memory Slot (Available)', 'WLAN', 'Ethernet', 'Audio Ports', 'Speaker',
       'Touchpad', 'Webcam', 'Style', 'Type', 'Usage', 'AC Adapter/Charger',
       'Dimensions (W x D x H)', 'Weight', 'Package Content',
       'Neural Processing Unit (NPU)', 'Display Features',
       'Op

In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 719 entries, 0 to 718
Data columns (total 87 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   title                         719 non-null    str    
 1   link                          719 non-null    str    
 2   current_price                 716 non-null    str    
 3   old_price                     126 non-null    str    
 4   discount                      99 non-null     str    
 5   rating                        94 non-null     float64
 6   rating_num                    94 non-null     float64
 7   shipping                      655 non-null    str    
 8   Best Seller Ranking           91 non-null     str    
 9   Series                        249 non-null    str    
 10  AI Features                   59 non-null     str    
 11  Operating System              351 non-null    str    
 12  Processor Name                267 non-null    str    
 13  Screen Size     

# refurbished or not

In [33]:
def get_refurbished(row):
    text = f"{row['title']} {row['Cosmetic Condition']}"
    if re.search(r'\b(refurbished|refurb|grade a|grade b|grade c)\b', text, re.IGNORECASE):
        return 1
    return 0

df['is_refurbished'] = df.apply(get_refurbished, axis=1)
print("Refurbished vs New Laptop Breakdown:")
print(df['is_refurbished'].value_counts().rename({0: 'New', 1: 'Refurbished'}))

Refurbished vs New Laptop Breakdown:
is_refurbished
New            439
Refurbished    280
Name: count, dtype: int64


# brand extraction

In [34]:
KNOWN_BRANDS = [
    'ASUS', 'Acer', 'Apple', 'Dell', 'DELL', 'HP', 'Lenovo', 'MSI', 'Microsoft', 
    'Samsung', 'Panasonic', 'GPD', 'THUNDEROBOT', 'NIAKUN', 'BiTECOOL', 
    'Auusda', 'Exsurf', 'NINGMEI', 'SAGAWHALE', 'Gigabyte', 'LG', 'Razer'
]

BRAND_MAP = {
    'dell': 'Dell', 'asus': 'Asus', 'hp': 'HP', 'lenovo': 'Lenovo', 'acer': 'Acer',
    'msi': 'MSI', 'apple': 'Apple', 'bitecool': 'BiTECOOL', 'auusda': 'Auusda',
    'exsurf': 'Auusda', 'ningmei': 'NINGMEI', 'sagawhale': 'SAGAWHALE', 'microsoft': 'Microsoft',
    'samsung': 'Samsung', 'panasonic': 'Panasonic', 'gpd': 'GPD', 'thunderobot': 'THUNDEROBOT',
    'gigabyte': 'Gigabyte', 'lg': 'LG', 'razer': 'Razer'
}

SERIES_BRAND_MAP = {
    r'\bthinkpad\b': 'Lenovo', r'\bideapad\b': 'Lenovo', r'\byoga\b': 'Lenovo', r'\blegion\b': 'Lenovo', r'\bloq\b': 'Lenovo',
    r'\blatitude\b': 'Dell', r'\bprecision\b': 'Dell', r'\binspiron\b': 'Dell', r'\bvostro\b': 'Dell', r'\balienware\b': 'Dell', r'\bxps\b': 'Dell',
    r'\belitebook\b': 'HP', r'\bprobook\b': 'HP', r'\bvictus\b': 'HP', r'\bpavilion\b': 'HP', r'\bomnibook\b': 'HP', r'\bzbook\b': 'HP', r'\benvy\b': 'HP', r'\bchromebook\b': 'HP',
    r'\bvivobook\b': 'Asus', r'\bzenbook\b': 'Asus', r'\brog\b': 'Asus', r'\btuf\b': 'Asus',
    r'\baspire\b': 'Acer', r'\bpredator\b': 'Acer', r'\bnitro\b': 'Acer', r'\bswift\b': 'Acer',
    r'\bkatana\b': 'MSI', r'\braider\b': 'MSI', r'\bstealth\b': 'MSI', r'\bprestige\b': 'MSI', r'\bventure\b': 'MSI',
    r'\bmacbook\b': 'Apple', r'\bsurface\b': 'Microsoft', r'\bgalaxy book\b': 'Samsung'
}

def clean_brand(row):
    b = str(row['Brand']).strip()
    if b and b.lower() != 'nan':
        return BRAND_MAP.get(b.lower(), b)
    title = str(row['title'])
    for kb in KNOWN_BRANDS:
        if re.search(r'\b' + re.escape(kb) + r'\b', title, re.IGNORECASE):
            return BRAND_MAP.get(kb.lower(), kb.title())
    for series_pat, brand in SERIES_BRAND_MAP.items():
        if re.search(series_pat, title, re.IGNORECASE):
            return brand
    first_word = title.split()[0].replace('Refurbished', '').strip()
    if first_word:
        return BRAND_MAP.get(first_word.lower(), first_word.capitalize())
    return 'Unknown'

df['brand'] = df.apply(clean_brand, axis=1)
print(f"Brand Nulls After Cleaning: {df['brand'].isnull().sum()}")
print(df['brand'].value_counts())

Brand Nulls After Cleaning: 0
brand
Lenovo         202
Dell           201
HP             176
Asus            40
Acer            36
Apple           16
Microsoft       13
MSI             12
BiTECOOL         4
Auusda           4
Samsung          2
Panasonic        2
Niakun           2
SAGAWHALE        1
NINGMEI          1
Chheart          1
KurieTim         1
GPD              1
GXMO             1
Hasee            1
Kurietim         1
THUNDEROBOT      1
Name: count, dtype: int64


# extracting series

In [35]:
# extracting laptop series (e.g. ThinkPad, VivoBook, EliteBook)
SERIES_MAP = {
    # Apple / Microsoft / Samsung — check compound names before the bare word
    r'\bmacbook pro\b': 'MacBook Pro', r'\bmacbook air\b': 'MacBook Air', r'\bmacbook\b': 'MacBook',
    r'\bsurface laptop\b': 'Surface Laptop', r'\bsurface pro\b': 'Surface Pro', r'\bsurface\b': 'Surface',
    r'\bgalaxy book\d?\s*(?:pro|ultra)?\b': 'Galaxy Book',
    # Lenovo
    r'\bthinkbook\b': 'ThinkBook', r'\bthinkpad\b': 'ThinkPad', r'\bideapad\b': 'IdeaPad',
    r'\blegion\b': 'Legion', r'\bloq\b': 'LOQ', r'\byoga\b': 'Yoga', r'\bslim\b': 'Slim',
    r'\bv[\s-]?series\b': 'V-Series', r'\bv1[0-9](?:\s|-|$)': 'V-Series',
    # Dell
    r'\bxps\b': 'XPS', r'\balienware\b': 'Alienware', r'\blatitude\b': 'Latitude',
    r'\bprecision\b': 'Precision', r'\bvostro\b': 'Vostro', r'\binspiron\b': 'Inspiron',
    r'\bdell pro\b': 'Dell Pro', r'\b(?:16|14)\s*plus\b': 'Dell Plus',
    # HP
    r'\belitebook\b': 'EliteBook', r'\bprobook\b': 'ProBook', r'\bzbook\b': 'ZBook',
    r'\bvictus\b': 'Victus', r'\bomnibook\b': 'OmniBook', r'\benvy\b': 'Envy',
    r'\bpavilion\b': 'Pavilion', r'\bchromebook\b': 'Chromebook',
    # Asus
    r'\brog\b': 'ROG', r'\btuf\b': 'TUF', r'\bzenbook\b': 'ZenBook', r'\bvivobook\b': 'VivoBook',
    # Acer
    r'\bpredator\b': 'Predator', r'\btravelmate\b': 'TravelMate', r'\bnitro\b': 'Nitro',
    r'\baspire\b': 'Aspire', r'\bswift\b': 'Swift',
    # MSI
    r'\bventurepro\b': 'VenturePro', r'\bventure\b': 'Venture', r'\braider\b': 'Raider',
    r'\bkatana\b': 'Katana', r'\bstealth\b': 'Stealth', r'\bprestige\b': 'Prestige',
    r'\bsummit\b': 'Summit', r'\bmodern\b': 'Modern',
}

def clean_series(row):
    title = str(row['title'])
    for pat, name in SERIES_MAP.items():
        if re.search(pat, title, re.IGNORECASE):
            return name
    # fall back to the raw 'Series' column when title has no known keyword
    raw = row.get('Series')
    if pd.notnull(raw):
        raw = str(raw).strip()
        for pat, name in SERIES_MAP.items():
            if re.search(pat, raw, re.IGNORECASE):
                return name
        # only trust it if it looks like a real series name, not a part/model number
        if raw and re.fullmatch(r'[A-Za-z][A-Za-z0-9\-\s\.]{1,24}', raw) and not raw.isdigit():
            return raw.title()
    return 'Other'

df['series'] = df.apply(clean_series, axis=1)

# current and old prices, discounts

In [36]:
df['current_price'] = df['current_price'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip()
df['current_price'] = pd.to_numeric(df['current_price'], errors='coerce')

df['old_price'] = df['old_price'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip()
df['old_price'] = pd.to_numeric(df['old_price'], errors='coerce')

df['discount_percent'] = df['discount'].astype(str).str.replace('%', '', regex=False).str.strip()
df['discount_percent'] = pd.to_numeric(df['discount_percent'], errors='coerce')

# Impute old price when missing but current_price & discount exist
mask = df['old_price'].isnull() & (df['discount_percent'] > 0) & df['current_price'].notnull()
df.loc[mask, 'old_price'] = np.round(df.loc[mask, 'current_price'] / (1 - df.loc[mask, 'discount_percent'] / 100), 2)
df['old_price'] = df['old_price'].fillna(df['current_price'])

# Calculate missing discount percent
mask = df['discount_percent'].isnull() & df['current_price'].notnull() & df['old_price'].notnull()
calc_disc = np.maximum(0, np.round((df['old_price'] - df['current_price']) / df['old_price'] * 100, 1))
df.loc[mask, 'discount_percent'] = calc_disc
df['discount_percent'] = df['discount_percent'].fillna(0)

df['current_price'] = df['current_price'].fillna(df['old_price'])

# shipping free or not

In [37]:
def clean_shipping(val):
    s = str(val).strip()
    if 'free' in s.lower():
        return 0.0, 1
    m = re.search(r'(\d+(?:\.\d+)?)', s)
    if m:
        return float(m.group(1)), 0
    return 0.0, 1

shipping_info = df['shipping'].apply(clean_shipping)
df['shipping_cost'] = [x[0] for x in shipping_info]
df['is_free_shipping'] = [x[1] for x in shipping_info]

# ram(memory) capacity and type

In [38]:
def extract_ram_gb(row):
    mem = str(row['Memory'])
    m = re.search(r'(\d+)\s*GB', mem, re.IGNORECASE)
    if m:
        val = int(m.group(1))
        if val in [4, 8, 12, 16, 24, 32, 40, 48, 64, 96, 128]:
            return val
    title = str(row['title'])
    m = re.search(r'(\d+)\s*GB\s*(?:DDR|LPDDR|RAM|Memory|onboard)', title, re.IGNORECASE)
    if m:
        val = int(m.group(1))
        if val in [4, 8, 12, 16, 24, 32, 40, 48, 64, 96, 128]:
            return val
    m = re.search(r'\b(4|8|12|16|24|32|40|48|64|96|128)\s*GB\b', title, re.IGNORECASE)
    if m:
        return int(m.group(1))
    return 16

def extract_ram_type(row):
    text = f"{row['Memory']} {row['Memory Spec']} {row['title']}"
    m = re.search(r'(LPDDR5X|LPDDR5|LPDDR4X|LPDDR4|LPDDR3|DDR5|DDR4|DDR3)', text, re.IGNORECASE)
    if m:
        return m.group(1).upper()
    return 'DDR4'

df['ram_capacity_gb'] = df.apply(extract_ram_gb, axis=1)
df['ram_type'] = df.apply(extract_ram_type, axis=1)

print("RAM Capacity Distribution (GB):")
print(df['ram_capacity_gb'].value_counts().sort_index())
print("\nRAM Type Breakdown:")
print(df['ram_type'].value_counts())

RAM Capacity Distribution (GB):
ram_capacity_gb
4      20
8      66
12      3
16    362
24     12
32    211
40      8
48      6
64     31
Name: count, dtype: int64

RAM Type Breakdown:
ram_type
DDR4       570
DDR5       108
LPDDR5X     25
DDR3         9
LPDDR5       3
LPDDR4       2
LPDDR3       1
LPDDR4X      1
Name: count, dtype: int64


# storage capacity and type

In [39]:
def extract_storage_gb(row):
    text = f"{row['SSD']} {row['HDD']} {row['Storage Spec']} {row['title']}"
    m = re.search(r'(\d+(?:\.\d+)?)\s*TB', text, re.IGNORECASE)
    if m:
        val = float(m.group(1))
        return int(val * 1024)
    m = re.search(r'(\d+)\s*GB\s*(?:SSD|NVMe|PCIe|SATA|HDD|Storage|UFS|eMMC)', text, re.IGNORECASE)
    if m:
        val = int(m.group(1))
        if val >= 64 and val != 1000:
            return val
    m = re.search(r'\b(128|256|512)\s*GB\b', text, re.IGNORECASE)
    if m:
        return int(m.group(1))
    return 512

def extract_storage_type(row):
    text = f"{row['SSD']} {row['HDD']} {row['Storage Spec']} {row['title']}"
    if re.search(r'\b(NVMe|PCIe)\b', text, re.IGNORECASE):
        return 'NVMe PCIe SSD'
    elif re.search(r'\bSATA\b', text, re.IGNORECASE):
        return 'SATA SSD'
    elif re.search(r'\bSSD\b', text, re.IGNORECASE):
        return 'SSD'
    elif re.search(r'\bHDD\b', text, re.IGNORECASE):
        return 'HDD'
    return 'SSD'

df['storage_capacity_gb'] = df.apply(extract_storage_gb, axis=1)
df['storage_type'] = df.apply(extract_storage_type, axis=1)

print("Storage Capacity Distribution (GB):")
print(df['storage_capacity_gb'].value_counts().sort_index())
print("\nStorage Type Breakdown:")
print(df['storage_type'].value_counts())

Storage Capacity Distribution (GB):
storage_capacity_gb
64        3
128      26
180       1
256     143
320       1
500       5
512     277
1024    232
2048     29
4096      2
Name: count, dtype: int64

Storage Type Breakdown:
storage_type
SSD              495
NVMe PCIe SSD    220
HDD                3
SATA SSD           1
Name: count, dtype: int64


# screen size, display resolution category and touchscreen 

In [40]:
def extract_screen_size(row):
    val = row['Screen Size']
    if pd.notnull(val):
        m = re.search(r'(\d+(?:\.\d+)?)', str(val))
        if m:
            s = float(m.group(1))
            if 10 <= s <= 20:
                return s
    title = str(row['title'])
    m = re.search(r'(\d{2}(?:\.\d)?)\s*(?:"|inch|in\b|-inch)', title, re.IGNORECASE)
    if m:
        s = float(m.group(1))
        if 10 <= s <= 20:
            return s
    return 15.6

def extract_resolution(row):
    res = f"{row['Resolution']} {row['Display Type']} {row['title']}"
    m = re.search(r'(\d{3,4})\s*x\s*(\d{3,4})', res, re.IGNORECASE)
    if m:
        w, h = int(m.group(1)), int(m.group(2))
        return f"{w}x{h}"
    if re.search(r'\b3K\b', res, re.IGNORECASE):
        return '2880x1800'
    elif re.search(r'\b4K\b|UHD', res, re.IGNORECASE):
        return '3840x2160'
    elif re.search(r'\bFHD\b|1080p', res, re.IGNORECASE):
        return '1920x1080'
    elif re.search(r'\bHD\b', res, re.IGNORECASE):
        return '1366x768'
    return '1920x1080'

def get_res_category(res):
    if '3840' in res or '2160' in res or '4K' in res:
        return '4K UHD'
    elif '2880' in res or '2560' in res or '3K' in res or '2K' in res or '1600' in res:
        return 'QHD / 3K'
    elif '1200' in res or '1920x1200' in res:
        return 'FHD+ (1200p)'
    elif '1080' in res or '1920x1080' in res or '1980' in res:
        return 'FHD (1080p)'
    elif '768' in res or '900' in res or '1366' in res:
        return 'HD'
    return 'FHD (1080p)'

df['screen_size_inches'] = df.apply(extract_screen_size, axis=1)
df['resolution'] = df.apply(extract_resolution, axis=1)
df['resolution_category'] = df['resolution'].apply(get_res_category)
df['is_touchscreen'] = df.apply(lambda r: 1 if re.search(r'Touch|Touchscreen', f"{r['Touchscreen']} {r['Display Type']} {r['title']}", re.IGNORECASE) else 0, axis=1)

print("Resolution Category Distribution:")
print(df['resolution_category'].value_counts())
print(f"\nTouchscreen Laptops: {df['is_touchscreen'].sum()} / {len(df)}")

Resolution Category Distribution:
resolution_category
FHD (1080p)     508
FHD+ (1200p)     94
HD               53
QHD / 3K         36
4K UHD           28
Name: count, dtype: int64

Touchscreen Laptops: 377 / 719


# processor(cpu - brand, series and model) and core count

In [41]:
def extract_cpu_info(row):
    text = f"{row['Processor Name']} {row['CPU Type']} {row['CPU']} {row['title']}"
    brand = 'Intel'
    if re.search(r'\b(AMD|Ryzen|Athlon)\b', text, re.IGNORECASE):
        brand = 'AMD'
    elif re.search(r'\b(Apple|MacBook|M1|M2|M3)\b', text, re.IGNORECASE):
        brand = 'Apple'
    elif re.search(r'\b(Snapdragon|Qualcomm)\b', text, re.IGNORECASE):
        brand = 'Snapdragon'
    elif re.search(r'\b(Intel|Core|Celeron|Pentium|Xeon)\b', text, re.IGNORECASE):
        brand = 'Intel'
        
    series = 'Other'
    m = re.search(r'Core Ultra \d', text, re.IGNORECASE)
    if m:
        series = m.group(0)
    else:
        m = re.search(r'Core i\d', text, re.IGNORECASE)
        if m:
            series = m.group(0)
        else:
            m = re.search(r'Ryzen AI \d', text, re.IGNORECASE)
            if m:
                series = m.group(0)
            else:
                m = re.search(r'Ryzen \d', text, re.IGNORECASE)
                if m:
                    series = m.group(0)
                elif re.search(r'Athlon', text, re.IGNORECASE):
                    series = 'Athlon'
                elif re.search(r'Pentium', text, re.IGNORECASE):
                    series = 'Pentium'
                elif re.search(r'Celeron', text, re.IGNORECASE):
                    series = 'Celeron'
                elif re.search(r'Processor N\d+|Intel N\d+|N150|N95|N5095', text, re.IGNORECASE):
                    series = 'Intel N-Series'
                else:
                    m = re.search(r'M[123](?:\s*(?:Pro|Max|Ultra))?', text, re.IGNORECASE)
                    if m:
                        series = m.group(0)

    model = ''
    patterns = [
        r'Intel Core Ultra \d \d+[A-Z]*',
        r'Core Ultra \d \d+[A-Z]*',
        r'i\d[- ]\d{4,5}[A-Z]*',
        r'Ryzen (?:AI )?\d \d{4}[A-Z]*',
        r'Ryzen AI \d \d+',
        r'Ryzen \d PRO \d{4}[A-Z]*',
        r'Athlon Silver \d+[A-Z]*',
        r'Pentium Gold \d+[A-Z]*',
        r'Celeron N\d+',
        r'N\d{3,4}'
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            model = m.group(0)
            break
    if not model:
        model = series

    return pd.Series([brand, series.title(), model])

def extract_cores(row):
    text = f"{row['Number of Cores']} {row['title']}"
    m = re.search(r'(\d+)\s*[- ]*(?:core|cores)\b', text, re.IGNORECASE)
    if m:
        return int(m.group(1))
    if re.search(r'Quad-core|4-core|4 Cores', text, re.IGNORECASE):
        return 4
    if re.search(r'Octa-core|8-core|8 Cores', text, re.IGNORECASE):
        return 8
    if re.search(r'Dual-core|2-core|2 Cores', text, re.IGNORECASE):
        return 2
    if re.search(r'Hexa-core|6-core|6 Cores', text, re.IGNORECASE):
        return 6
    if re.search(r'10-core', text, re.IGNORECASE):
        return 10
    if re.search(r'12-core', text, re.IGNORECASE):
        return 12
    if re.search(r'14-core', text, re.IGNORECASE):
        return 14
    if re.search(r'16-core', text, re.IGNORECASE):
        return 16
    return None

df[['cpu_brand', 'cpu_series', 'cpu_model']] = df.apply(extract_cpu_info, axis=1)
df['cpu_cores'] = df.apply(extract_cores, axis=1)

print("CPU Vendor Distribution:")
print(df['cpu_brand'].value_counts())
print("\nTop CPU Series:")
print(df['cpu_series'].value_counts().head(8))
print("CPU Models:")
print(df['cpu_model'].value_counts())

CPU Vendor Distribution:
cpu_brand
Intel         529
AMD           152
Snapdragon     21
Apple          17
Name: count, dtype: int64

Top CPU Series:
cpu_series
Core I7         152
Core I5         142
Other           101
Core Ultra 7     82
Ryzen 7          66
Ryzen 5          42
Core Ultra 5     25
Ryzen Ai 7       20
Name: count, dtype: int64
CPU Models:
cpu_model
Other                      60
i7-1355U                   32
i7-1185G                   28
Ryzen 7 7730U              22
i5-1145G                   19
                           ..
Intel Core Ultra 5 235U     1
Intel Core Ultra 5 115U     1
i7 10610U                   1
i7 10850H                   1
Ryzen 5 4000                1
Name: count, Length: 202, dtype: int64


# graphics card(gpu - brand, type) and operating system

In [42]:
def extract_gpu_info(row):
    text = f"{row['GPU/VPU']} {row['Graphic Type']} {row['Graphics Card']} {row['title']}"
    brand = 'Integrated / Other'
    if re.search(r'NVIDIA|GeForce|RTX|GTX|Quadro|T550|T1200|T2000', text, re.IGNORECASE):
        brand = 'NVIDIA'
    elif re.search(r'AMD|Radeon|Vega', text, re.IGNORECASE):
        brand = 'AMD'
    elif re.search(r'Intel|Iris|Arc|UHD|HD Graphics', text, re.IGNORECASE):
        brand = 'Intel'
    elif re.search(r'Apple', text, re.IGNORECASE):
        brand = 'Apple'

    gpu_type = 'Integrated'
    if brand in ['NVIDIA'] or re.search(r'Dedicated|GeForce|RTX|GTX|Quadro', text, re.IGNORECASE):
        gpu_type = 'Dedicated'

    model = 'Integrated Graphics'
    patterns = [
        r'RTX \d{4}(?:\s*Laptop GPU)?',
        r'Quadro T\d{4}',
        r'T\d{3,4}',
        r'Radeon 780M',
        r'Radeon RX Vega \d',
        r'Radeon Graphics',
        r'Arc 140V|Arc Graphics',
        r'Iris Xe',
        r'UHD Graphics'
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            model = m.group(0)
            break
            
    return pd.Series([brand, gpu_type, model])

def clean_os(val, title):
    text = f"{val} {title}"
    if re.search(r'Windows 11 Pro', text, re.IGNORECASE):
        return 'Windows 11 Pro'
    elif re.search(r'Windows 11 Home', text, re.IGNORECASE):
        return 'Windows 11 Home'
    elif re.search(r'Windows 11', text, re.IGNORECASE):
        return 'Windows 11 Home'
    elif re.search(r'Windows 10 Pro', text, re.IGNORECASE):
        return 'Windows 10 Pro'
    elif re.search(r'Windows 10', text, re.IGNORECASE):
        return 'Windows 10 Home'
    elif re.search(r'Chrome', text, re.IGNORECASE):
        return 'Chrome OS'
    elif re.search(r'Mac', text, re.IGNORECASE):
        return 'macOS'
    return 'Windows 11 Home'

df[['gpu_brand', 'gpu_type', 'gpu_model']] = df.apply(extract_gpu_info, axis=1)
df['operating_system'] = df.apply(lambda r: clean_os(r['Operating System'], r['title']), axis=1)

print("GPU Vendor Distribution:")
print(df['gpu_brand'].value_counts())
print("\nGPU Type (Integrated vs Dedicated):")
print(df['gpu_type'].value_counts())
print("\nOperating System Distribution:")
print(df['operating_system'].value_counts())

GPU Vendor Distribution:
gpu_brand
Intel                 434
AMD                   146
Integrated / Other     75
NVIDIA                 52
Apple                  12
Name: count, dtype: int64

GPU Type (Integrated vs Dedicated):
gpu_type
Integrated    665
Dedicated      54
Name: count, dtype: int64

Operating System Distribution:
operating_system
Windows 11 Pro     385
Windows 11 Home    265
Windows 10 Pro      33
macOS               16
Chrome OS           10
Windows 10 Home     10
Name: count, dtype: int64


# rating, backlit keyboard, ai ready

In [43]:
df['is_ai_pc'] = df.apply(lambda r: 1 if re.search(r'Copilot\+|AI Ready|AI PC|NPU', f"{r['AI Features']} {r['Neural Processing Unit (NPU)']} {r['title']}", re.IGNORECASE) else 0, axis=1)
df['has_backlit_keyboard'] = df.apply(lambda r: 1 if re.search(r'Backlit', f"{r['Backlit Keyboard']} {r['Keyboard']} {r['title']}", re.IGNORECASE) else 0, axis=1)
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df['rating_num'] = pd.to_numeric(df['rating_num'], errors='coerce').fillna(0).astype(int)


# transfer cleaned columns to cleaned_df

In [44]:
final_cols = [
    'title', 'brand', 'series', 'is_refurbished', 'current_price', 'old_price', 'discount_percent',
    'shipping_cost', 'is_free_shipping', 'rating', 'rating_num', 'operating_system',
    'screen_size_inches', 'resolution', 'resolution_category', 'is_touchscreen',
    'ram_capacity_gb', 'ram_type', 'storage_capacity_gb', 'storage_type',
    'cpu_brand', 'cpu_series', 'cpu_model', 'cpu_cores',
    'gpu_brand', 'gpu_type', 'gpu_model', 'is_ai_pc', 'has_backlit_keyboard', 'link'
]

cleaned_df = df[final_cols].copy()

print(f"Final Shape: {cleaned_df.shape[0]} rows x {cleaned_df.shape[1]} columns\n")
print("--- NULL VALUE CHECK (FINAL DATASET) ---")
print(cleaned_df.isnull().sum())

Final Shape: 719 rows x 30 columns

--- NULL VALUE CHECK (FINAL DATASET) ---
title                     0
brand                     0
series                    0
is_refurbished            0
current_price             0
old_price                 0
discount_percent          0
shipping_cost             0
is_free_shipping          0
rating                  625
rating_num                0
operating_system          0
screen_size_inches        0
resolution                0
resolution_category       0
is_touchscreen            0
ram_capacity_gb           0
ram_type                  0
storage_capacity_gb       0
storage_type              0
cpu_brand                 0
cpu_series                0
cpu_model                 0
cpu_cores               457
gpu_brand                 0
gpu_type                  0
gpu_model                 0
is_ai_pc                  0
has_backlit_keyboard      0
link                      0
dtype: int64


# export the cleaned data to csv

In [45]:
BASE_DIR = Path(r"D:\project\laptop_pricing_intelligence_pipeline")

output_dir = BASE_DIR / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

cleaned_df.to_csv(output_dir / "try.csv", index=False)

In [46]:
cleaned_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 719 entries, 0 to 718
Data columns (total 30 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   title                 719 non-null    str    
 1   brand                 719 non-null    str    
 2   series                719 non-null    str    
 3   is_refurbished        719 non-null    int64  
 4   current_price         719 non-null    float64
 5   old_price             719 non-null    float64
 6   discount_percent      719 non-null    float64
 7   shipping_cost         719 non-null    float64
 8   is_free_shipping      719 non-null    int64  
 9   rating                94 non-null     float64
 10  rating_num            719 non-null    int64  
 11  operating_system      719 non-null    str    
 12  screen_size_inches    719 non-null    float64
 13  resolution            719 non-null    str    
 14  resolution_category   719 non-null    str    
 15  is_touchscreen        719 non-null

In [47]:
cleaned_df.describe()

,is_refurbished,current_price,old_price,discount_percent,shipping_cost,is_free_shipping,rating,rating_num,screen_size_inches,is_touchscreen,ram_capacity_gb,storage_capacity_gb,cpu_cores,is_ai_pc,has_backlit_keyboard
count,719.00000,719.000000,719.000000,719.000000,719.000000,719.000000,94.000000,719.000000,719.000000,719.000000,719.000000,719.000000,262.000000,719.000000,719.000000
mean,0.38943,873.486453,902.805202,2.877330,0.501446,0.966620,4.191489,2.041725,15.013213,0.524339,22.347705,681.646732,56.190840,0.116829,0.343533
std,0.48796,696.421193,719.129313,8.217154,2.848504,0.179751,1.108099,12.335759,1.137843,0.499755,12.663476,454.304234,564.165761,0.321440,0.475218
min,0.00000,89.000000,99.000000,0.000000,0.000000,0.000000,1.000000,0.000000,11.600000,0.000000,4.000000,64.000000,1.000000,0.000000,0.000000
25%,0.00000,411.990000,439.990000,0.000000,0.000000,1.000000,4.000000,0.000000,14.000000,0.000000,16.000000,512.000000,4.000000,0.000000,0.000000
50%,0.00000,699.990000,719.000000,0.000000,0.000000,1.000000,4.500000,0.000000,15.600000,1.000000,16.000000,512.000000,8.000000,0.000000,0.000000
75%,1.00000,1129.000000,1149.495000,0.000000,0.000000,1.000000,5.000000,0.000000,15.600000,1.000000,32.000000,1024.000000,10.000000,0.000000,1.000000
max,1.00000,6531.990000,6531.990000,54.000000,19.990000,1.000000,5.000000,130.000000,18.000000,1.000000,64.000000,4096.000000,7480.000000,1.000000,1.000000


In [48]:
cleaned_df.isnull().sum()

title                     0
brand                     0
series                    0
is_refurbished            0
current_price             0
old_price                 0
discount_percent          0
shipping_cost             0
is_free_shipping          0
rating                  625
rating_num                0
operating_system          0
screen_size_inches        0
resolution                0
resolution_category       0
is_touchscreen            0
ram_capacity_gb           0
ram_type                  0
storage_capacity_gb       0
storage_type              0
cpu_brand                 0
cpu_series                0
cpu_model                 0
cpu_cores               457
gpu_brand                 0
gpu_type                  0
gpu_model                 0
is_ai_pc                  0
has_backlit_keyboard      0
link                      0
dtype: int64

In [49]:
cleaned_df.duplicated().sum()

np.int64(0)